# Qwen single-agent TEC tool-calling experiment

This notebook runs the first Colab-side LLM smoke experiment for the project: Qwen as a single agent using textual `<tool_call>` / `<final_answer>` tags, the project parser, and the local MCP-like tool client.

## 1. Environment setup

Run this in Google Colab. The notebook does not download IONEX files and does not store model or processed data files in Git.

In [ ]:
# Clean Colab setup for Qwen.
# Run this cell first, then restart runtime before importing transformers.

!pip uninstall -y transformers scikit-learn scipy
!pip install -U accelerate bitsandbytes torchvision pillow sentencepiece protobuf safetensors huggingface_hub pandas pyarrow pydantic python-dateutil

In [ ]:
!pip install "transformers @ git+https://github.com/huggingface/transformers.git@main"

IMPORTANT:

After the installation cell finishes, restart the Colab runtime:

Runtime -> Restart runtime

Then continue from the import check cell below.
Do not import transformers before restarting the runtime.

In [ ]:
import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
print("transformers Qwen imports OK")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/tec_agent_project")
REPO_URL = "https://github.com/ilyakosilov/tec_agent_project.git"

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Working directory:", Path.cwd())
print("src on path:", SRC_DIR in map(Path, sys.path))

## 2. Data setup

Expected processed dataset path inside the cloned repo:

`data/processed/tec_regions_2024_03_hourly.parquet`

This notebook does not rebuild IONEX data. It expects the processed parquet file produced by `notebooks/01_build_tec_dataset.ipynb`.

If you keep this parquet in Google Drive, set `MOUNT_DRIVE = True` and update `DRIVE_PARQUET_PATH`.

In [ ]:
import shutil

DATASET_PATH = PROJECT_DIR / "data" / "processed" / "tec_regions_2024_03_hourly.parquet"

MOUNT_DRIVE = False
DRIVE_PARQUET_PATH = Path("/content/drive/MyDrive/tec_regions_2024_03_hourly.parquet")

if MOUNT_DRIVE and not DATASET_PATH.exists():
    from google.colab import drive

    drive.mount("/content/drive")
    if DRIVE_PARQUET_PATH.exists():
        DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(DRIVE_PARQUET_PATH, DATASET_PATH)

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Processed dataset not found: {DATASET_PATH}\n"
        "Upload or copy tec_regions_2024_03_hourly.parquet to "
        "data/processed/ before running the LLM experiment."
    )

print("Dataset path:", DATASET_PATH)

## 3. Imports

In [ ]:
import json

from tec_agents.agents.llm_single_agent import LLMSingleAgent
from tec_agents.data.datasets import get_dataset_summary, register_dataset
from tec_agents.eval.task_set import build_default_research_tasks
from tec_agents.llm.local_qwen import LocalQwenChatModel
from tec_agents.mcp.client import LocalMCPClient
from tec_agents.mcp.server import build_local_mcp_server

## 4. Register dataset

In [ ]:
register_dataset(
    dataset_ref="default",
    path=DATASET_PATH,
    file_format="parquet",
)

summary = get_dataset_summary("default")
summary

## 5. Initialize MCP-like tools

In [ ]:
server = build_local_mcp_server(run_id="qwen_single_agent_colab_smoke")
client = LocalMCPClient(server)

print("Available tools:")
for name in client.list_tool_names():
    print("-", name)

## 6. Load Qwen

The previous exploratory notebook used `Qwen/Qwen3.5-4B`, 4-bit NF4 quantization, `device_map="auto"`, `torch.bfloat16`, `trust_remote_code=True`, `max_input_length=4096`, deterministic generation with `do_sample=False`, and optional Hugging Face login through Colab `userdata`. Change `MODEL_NAME` here if your Hugging Face checkpoint name differs.

In [ ]:
USE_HF_TOKEN = False
HF_TOKEN_SECRET_NAME = "HF_read_token"

if USE_HF_TOKEN:
    from google.colab import userdata
    from huggingface_hub import login

    hf_token = userdata.get(HF_TOKEN_SECRET_NAME)
    if not hf_token:
        raise ValueError(f"Colab secret {HF_TOKEN_SECRET_NAME!r} is empty or missing.")
    login(token=hf_token)
    print("Logged in to Hugging Face.")

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-4B"
LOAD_IN_4BIT = False
LOAD_IN_8BIT = False
TORCH_DTYPE = "float16"
MAX_INPUT_TOKENS = 4096

model = LocalQwenChatModel(
    model_name=MODEL_NAME,
    device_map="auto",
    torch_dtype=TORCH_DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    load_in_8bit=LOAD_IN_8BIT,
    trust_remote_code=True,
    max_input_tokens=MAX_INPUT_TOKENS,
)

## 7. Run one task

In [ ]:
query = "Find high TEC intervals for midlat_europe in March 2024 with q=0.9"

agent = LLMSingleAgent(
    model=model,
    client=client,
    max_steps=12,
    max_tool_calls=8,
    max_parse_retries=2,
    max_tool_retries=2,
    temperature=0.0,
)

result = agent.run(query)

print("success:", result.success)
print("answer:\n", result.answer)
print("tool_sequence:", result.tool_sequence)
print("parse_error_count:", result.parse_error_count)
print("invalid_json_count:", result.invalid_json_count)
print("unknown_format_count:", result.unknown_format_count)
print("repair_attempt_count:", result.repair_attempt_count)
print("error_message:", result.error_message)

In [ ]:
print("Raw model outputs")
print("=" * 80)
for i, raw in enumerate(result.raw_model_outputs, start=1):
    print(f"\n--- output {i} ---")
    print(raw)

print("\nTrace")
print("=" * 80)
print(json.dumps(result.trace, ensure_ascii=False, indent=2))

## 8. Compare with expected sequence

In [ ]:
expected_sequence = [
    "tec_get_timeseries",
    "tec_compute_high_threshold",
    "tec_detect_high_intervals",
]

sequence_match = result.tool_sequence == expected_sequence
print("expected:", expected_sequence)
print("actual:  ", result.tool_sequence)
print("sequence_match:", sequence_match)

## 9. Optional: run a small task subset

Leave `RUN_SUBSET = False` for the first smoke run. Set it to `True` when the one-task run is stable.

In [ ]:
RUN_SUBSET = False

subset_queries = [
    "Find high TEC intervals for midlat_europe in March 2024 with q=0.9",
    "Find high TEC intervals for highlat_north in March 2024 with q=0.9",
    "Compare TEC statistics for midlat_europe and highlat_north in March 2024",
]

subset_results = []

if RUN_SUBSET:
    for idx, subset_query in enumerate(subset_queries, start=1):
        subset_server = build_local_mcp_server(run_id=f"qwen_single_agent_subset_{idx}")
        subset_client = LocalMCPClient(subset_server)
        subset_agent = LLMSingleAgent(
            model=model,
            client=subset_client,
            max_steps=16,
            max_tool_calls=12,
            max_parse_retries=2,
            max_tool_retries=2,
            temperature=0.0,
        )
        subset_result = subset_agent.run(subset_query)
        subset_results.append({
            "query": subset_query,
            "result": subset_result.to_dict(),
        })
        print(idx, subset_query)
        print("  success:", subset_result.success)
        print("  tool_sequence:", subset_result.tool_sequence)
        print("  parse_errors:", subset_result.parse_error_count)

## 10. Save results

In [ ]:
output_path = PROJECT_DIR / "outputs" / "metrics" / "qwen_single_agent_smoke_colab.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

payload = {
    "model_name": MODEL_NAME,
    "task_query": query,
    "result": result.to_dict(),
    "raw_model_outputs": result.raw_model_outputs,
    "tool_sequence": result.tool_sequence,
    "sequence_match": sequence_match,
    "parse_counters": {
        "parse_error_count": result.parse_error_count,
        "invalid_json_count": result.invalid_json_count,
        "unknown_format_count": result.unknown_format_count,
        "repair_attempt_count": result.repair_attempt_count,
    },
    "success": result.success,
    "subset_results": subset_results,
}

with output_path.open("w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("Saved:", output_path)